In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Dataset Overview
<p>The American Sign Language (ASL) Alphabet Dataset contains images of hand gestures representing the 26 letters (A-Z) of the English alphabet along with 3 additional classes: SPACE, DELETE and NOTHING.</p>
<p>The training dataset contains 87,000 images of size 200×200 pixels
while the test dataset contains 29 images</p>

The dataset is used for building gesture recognition systems to assist communication for deaf or hard-of-hearing individuals enabling applications like gesture-based typing, human-computer interaction and assistive technologies.

<p><b></b>Feature Analysis:</b></p>

**Predictor Variables:** Image pixels (RGB channels).

**Target Variable:** Class label representing the letter or gesture (A-Z, SPACE, DELETE, NOTHING).

**Data Quality:**
Images are mostly uniform in size but may vary in hand orientation, background and lighting.

Dataset is relatively balanced though the 3 special classes may have fewer images.

Some images may include background noise or subtle variations in hand shape.

## Step 0: Import Required Libraries

In [ ]:
# Core Python libraries
# Core Python libraries used for data handling, visualization, and randomness

import os  
# Imports the OS module to interact with the operating system
# Used for file handling, directory navigation, and environment variables

import numpy as np  
# Imports NumPy library for numerical and mathematical operations
# Commonly used for arrays, matrices, and linear algebra

import matplotlib.pyplot as plt  
# Imports pyplot from Matplotlib for creating plots and graphs
# Used for visualizing data such as line plots, bar charts, etc.

import seaborn as sns  
# Imports Seaborn for advanced and attractive statistical visualizations
# Built on top of Matplotlib

import random  
# Imports the random module to generate random numbers and random selections
# Useful for simulations, sampling, and shuffling data


In [ ]:
# TensorFlow / Keras libraries
# Import Sequential model to build the CNN layer by layer
from tensorflow.keras.models import Sequential

# Import Convolutional layer to extract spatial features from images
from tensorflow.keras.layers import Conv2D

# Import MaxPooling layer to reduce spatial dimensions and computation
from tensorflow.keras.layers import MaxPooling2D

# Import Flatten layer to convert 2D feature maps into a 1D vector
from tensorflow.keras.layers import Flatten

# Import Dense layer for fully connected neural network layers
from tensorflow.keras.layers import Dense

# Import Dropout layer to prevent overfitting by randomly disabling neurons
from tensorflow.keras.layers import Dropout

# Import BatchNormalization to stabilize and speed up model training
from tensorflow.keras.layers import BatchNormalization

# Import Adam optimizer for efficient gradient-based optimization
from tensorflow.keras.optimizers import Adam

# Import EarlyStopping to stop training when validation performance stops improving
from tensorflow.keras.callbacks import EarlyStopping

# Import ReduceLROnPlateau to reduce learning rate when validation loss plateaus
from tensorflow.keras.callbacks import ReduceLROnPlateau

# Import TensorFlow core library
import tensorflow as tf

# for augmenttation
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Suppress unnecessary TensorFlow warning and info messages for clean output
tf.get_logger().setLevel('ERROR')

In [ ]:
# For evaluation
from sklearn.metrics import classification_report, confusion_matrix

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

### Reproducibility Setup

To ensure consistent and repeatable experimental results, random seeds are fixed for:
- NumPy
- TensorFlow
- Python's built-in random module

Setting these seeds ensures that data shuffling, weight initialization, and augmentation behave identically across multiple executions of the notebook, which is essential for scientific reproducibility and fair evaluation.


In [ ]:
# Set the NumPy random seed to ensure reproducibility of NumPy operations
# This guarantees that random number generation (e.g., shuffling, sampling) is consistent
np.random.seed(42)

# Set TensorFlow's random seed for reproducible deep learning results
# Ensures consistent weight initialization and training behavior across runs
tf.random.set_seed(42)

# Set Python's built-in random seed
# Controls randomness in standard Python operations such as random selection
random.seed(42)


## Step 1: Dataset Overview & EDA

In [ ]:

# Import the OS module to interact with the operating system
import os

# Define the correct dataset directory path
# This path points to the main folder that contains all 29 class subfolders
dataset_dir = "/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train"

# Retrieve the names of all subdirectories inside the dataset directory
# Each subdirectory represents one class (A–Z, SPACE, DELETE, NOTHING)
classes = [
    d for d in os.listdir(dataset_dir)          # List all items in the dataset directory
    if os.path.isdir(os.path.join(dataset_dir, d))  # Keep only directories (ignore files)
]

# Print the total number of detected classes in the dataset
print("Number of classes:", len(classes))

# Print the names of all class folders
print("Classes:", classes)


In [ ]:
# Count images per class
class_counts = {cls: len(os.listdir(os.path.join(dataset_dir, cls))) for cls in classes}


## Step 2: Sample Images


To better understand the dataset, one representative image from each class is visualized.

**Purpose of this step:**
- Verify that images are correctly loaded from their respective class folders
- Visually inspect hand gesture patterns for different ASL alphabets
- Confirm image resolution, background consistency, and gesture clarity

**Observations:**
- Each image represents a distinct American Sign Language gesture
- Images are RGB and have uniform dimensions (200×200 pixels)
- Backgrounds are relatively consistent, which is beneficial for CNN training

This visualization validates the dataset structure and confirms its suitability for deep learning–based image classification.

In [ ]:
# Create a new matplotlib figure with a wide layout to display multiple images
plt.figure(figsize=(16, 8))

# Loop over each class name along with its index
for i, cls in enumerate(classes):
    
    # Construct the path to the current class directory
    class_dir = os.path.join(dataset_dir, cls)
    
    # Select the first image file from the class directory
    # This is used as a representative sample for that class
    img_path = os.path.join(class_dir, os.listdir(class_dir)[0])
    
    # Read the image file into a NumPy array
    img = plt.imread(img_path)
    
    # Create a subplot grid (4 rows × 8 columns) and place the image at position i+1
    plt.subplot(4, 8, i + 1)
    
    # Display the image
    plt.imshow(img)
    
    # Set the title of the subplot as the class label
    plt.title(cls)
    
    # Turn off axis ticks and labels for a cleaner visualization
    plt.axis("off")

# Automatically adjust subplot spacing to prevent overlap
plt.tight_layout()

# Render the figure on screen
plt.show()


## Step 3: Train–Validation–Test Split & Preprocessing

#### Dataset Split

Training Set: 80% of images

Validation Set: 20% of images (created using validation_split=0.2)

Test Set: Loaded separately without augmentation

This split allows reliable performance monitoring while preventing data leakage.

####  image preprocessing  
Rescaling: All images were normalized by dividing pixel values by 255, converting them to the range [0,1].

Resizing: Images were resized to 128 × 128 pixels for consistent CNN input.

Label Encoding: Class labels were automatically converted into one-hot encoded vectors for multi-class classification.

#### Data Augmentation (Training Only)

To improve generalization and reduce overfitting, the following augmentations were applied to the training set:

Random rotation (±15°)

Horizontal and vertical shifts (±10%)

Zooming (±10%)

Horizontal flipping

No augmentation was applied to validation or test data to ensure fair evaluation

In [ ]:
# Training data generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,             # Normalize pixel values from [0,255] to [0,1] for faster convergence
    rotation_range=15,          # Randomly rotate images within 15 degrees to make model rotation-invariant
    width_shift_range=0.1,      # Randomly shift images horizontally by up to 10% of total width
    height_shift_range=0.1,     # Randomly shift images vertically by up to 10% of total height
    zoom_range=0.1,             # Randomly zoom in/out up to 10% to improve scale invariance
    horizontal_flip=True,       # Randomly flip images horizontally for data diversity
    validation_split=0.2        # Reserve 20% of data for validation set
)

In [ ]:
# Training generator: loads training images from the directory with applied augmentations
train_generator = train_datagen.flow_from_directory(
    dataset_dir,                # Path to dataset directory containing class subfolders
    target_size=(128,128),       # Resize all images to 128x128 pixels for CNN input
    batch_size=64,               # Number of images per batch during training
    class_mode='categorical',    # Multi-class classification (one-hot encoded labels)
    subset='training',           # Use the training portion (80%) defined by validation_split
    shuffle=True                 # Shuffle images each epoch for better generalization
)

### Validation Data Generator

This step creates a validation data generator using Keras' `flow_from_directory` method.  
The generator automatically reads images from class-labeled folders and applies the same preprocessing steps as the training data.

Key points:
- Images are resized to **128×128** pixels for CNN compatibility.
- **Categorical labels** are used for multi-class classification (29 ASL gestures).
- **20% of the dataset** is reserved for validation using `validation_split`.
- Shuffling is disabled to ensure correct alignment between predictions and true labels during evaluation.


In [ ]:
# Validation generator: loads validation images without shuffling
val_generator = train_datagen.flow_from_directory(
    dataset_dir,                # Same dataset directory
    target_size=(128,128),      # Resize images to 128x128
    batch_size=64,              # Number of images per batch during validation
    class_mode='categorical',   # Multi-class classification
    subset='validation',        # Use the validation portion (20%) defined by validation_split
    shuffle=False               # No shuffling for consistent evaluation metrics
)

In [ ]:
# Test data generator (NO augmentation)
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    dataset_dir,              # or separate test directory
    target_size=(128, 128),
    batch_size=64,
    class_mode='categorical',
    shuffle=False             # IMPORTANT for evaluation
)


## Step 4: Build CNN Model from Scratch
- 3 convolutional blocks with increasing filters (32 → 64 → 128)
- Batch Normalization for stable training
- Dropout to reduce overfitting
- Fully connected layers with softmax output for 29 classes

In [ ]:
# Initialize a Sequential model which allows layers to be added one after another
model = Sequential()

# =======================
# Convolutional Block 1
# =======================

# Add a 2D convolution layer with 32 filters of size 3×3
# ReLU activation introduces non-linearity
# 'same' padding preserves spatial dimensions
# input_shape defines the shape of input images (128x128 RGB)
model.add(Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(128,128,3)))

# Normalize activations to improve training stability and convergence speed
model.add(BatchNormalization())

# Downsample feature maps by a factor of 2 to reduce spatial dimensions
model.add(MaxPooling2D(2,2))

# Randomly drop 25% of neurons to reduce overfitting
model.add(Dropout(0.25))

# =======================
# Convolutional Block 2
# =======================

# Add a deeper convolution layer with 64 filters to capture more complex patterns
model.add(Conv2D(64, (3,3), activation='relu', padding='same'))

# Apply batch normalization to stabilize gradients
model.add(BatchNormalization())

# Reduce spatial dimensions while retaining important features
model.add(MaxPooling2D(2,2))

# Apply dropout regularization to prevent co-adaptation of neurons
model.add(Dropout(0.25))

# =======================
# Convolutional Block 3
# =======================

# Add an even deeper convolution layer with 128 filters
# This layer captures high-level spatial features such as hand shapes
model.add(Conv2D(128, (3,3), activation='relu', padding='same'))

# Normalize outputs to improve learning efficiency
model.add(BatchNormalization())

# Further reduce feature map dimensions
model.add(MaxPooling2D(2,2))

# Dropout to control overfitting at deeper levels
model.add(Dropout(0.25))

# =======================
# Fully Connected Layers
# =======================

# Flatten 3D feature maps into a 1D feature vector
model.add(Flatten())

# Add a dense layer with 256 neurons for high-level feature learning
model.add(Dense(256, activation='relu'))

# Apply batch normalization to dense layer
model.add(BatchNormalization())

# Drop 50% of neurons to strongly reduce overfitting
model.add(Dropout(0.5))

# Add another dense layer with 128 neurons for refined feature representation
model.add(Dense(128, activation='relu'))

# Normalize dense layer activations
model.add(BatchNormalization())

# Apply dropout to further regularize the model
model.add(Dropout(0.5))

# =======================
# Output Layer
# =======================

# Final dense layer with neurons equal to the number of classes
# Softmax converts logits into class probabilities
model.add(Dense(len(classes), activation='softmax'))

# =======================
# Compile the Model
# =======================

# Compile the model using Adam optimizer for efficient gradient descent
# Categorical crossentropy is used for multi-class classification
# Accuracy metric tracks classification performance
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)



In [ ]:
# Display a summary of the model architecture and parameters
model.summary()


## Step 5: Train the Model

### Training Optimization Callbacks

To improve training efficiency and prevent overfitting, two Keras callbacks are used:

**EarlyStopping**
- Monitors validation loss
- Stops training when no improvement is observed for 5 epochs
- Restores the model weights from the best-performing epoch

**ReduceLROnPlateau**
- Monitors validation accuracy
- Reduces the learning rate by half when performance stagnates
- Helps the optimizer converge to a better minimum

These callbacks ensure faster convergence, improved generalization, and optimal use of training resources.



In [ ]:
# Create an EarlyStopping callback to stop training when validation loss stops improving
early_stop = EarlyStopping(
    
    # Metric to monitor during training
    monitor='val_loss',
    
    # Number of consecutive epochs with no improvement before stopping training
    patience=5,
    
    # Restore model weights from the epoch with the best validation loss
    restore_best_weights=True
)

# Create a ReduceLROnPlateau callback to decrease learning rate when validation accuracy plateaus
reduce_lr = ReduceLROnPlateau(
    
    # Metric to monitor for learning rate reduction
    monitor='val_accuracy',
    
    # Factor by which the learning rate will be reduced (new_lr = lr * factor)
    factor=0.5,
    
    # Number of epochs with no improvement before reducing the learning rate
    patience=3,
    
    # Minimum learning rate allowed to prevent extremely small updates
    min_lr=1e-6,
    
    # Print a message whenever the learning rate is reduced
    verbose=1
)



The model is trained using an augmented image generator to improve generalization.
Each epoch consists of multiple mini-batch updates generated by `train_generator`.

**Key Training Strategies:**
- **EarlyStopping** prevents overfitting by stopping training when validation loss no longer improves.
- **ReduceLROnPlateau** dynamically lowers the learning rate to fine-tune the model.
- Validation performance is evaluated after each epoch to monitor generalization.

This training setup ensures efficient convergence and stable performance on unseen data.

In [ ]:
# Train the CNN model using augmented training data
history = model.fit(
    
    # Training data generator that yields batches of augmented images and labels
    train_generator,
    
    # Total number of times the model will iterate over the entire training dataset
    epochs=30,
    
    # Validation data generator used to evaluate the model after each epoch
    validation_data=val_generator,
    
    # List of callbacks to control training behavior dynamically
    callbacks=[
        early_stop,   # Stops training early if validation loss stops improving
        reduce_lr     # Reduces learning rate when validation accuracy plateaus
    ]
)


## Step 6: Plot Training History

In [ ]:
# Accuracy score
# Create a new figure for plotting with a custom size (width=12, height=5 inches)
plt.figure(figsize=(12,5))

# Define the first subplot in a 1x2 grid, selecting the first plot (left side)
plt.subplot(1,2,1)

# Plot training accuracy over epochs using the 'accuracy' values from the history object
plt.plot(history.history['accuracy'], label='Train Acc')

# Plot validation accuracy over epochs using the 'val_accuracy' values from the history object
plt.plot(history.history['val_accuracy'], label='Val Acc')

# Set the title of the plot
plt.title('Training vs Validation Accuracy')

# Label the x-axis as 'Epoch'
plt.xlabel('Epoch')

# Label the y-axis as 'Accuracy'
plt.ylabel('Accuracy')

# Add a legend to differentiate between training and validation curves
plt.legend()

# Add a light grid to improve readability, with transparency of 0.3
plt.grid(alpha=0.3)


In [ ]:
# Loss curve
# Define the second subplot in a 1x2 grid, selecting the second plot (right side)
plt.subplot(1,2,2)

# Plot training loss over epochs using the 'loss' values from the history object
plt.plot(history.history['loss'], label='Train Loss')

# Plot validation loss over epochs using the 'val_loss' values from the history object
plt.plot(history.history['val_loss'], label='Val Loss')

# Set the title of the plot
plt.title('Training vs Validation Loss')

# Label the x-axis as 'Epoch'
plt.xlabel('Epoch')

# Label the y-axis as 'Loss'
plt.ylabel('Loss')

# Add a legend to differentiate between training and validation curves
plt.legend()

# Add a light grid to improve readability, with transparency of 0.3
plt.grid(alpha=0.3)

# Display the figure with both accuracy and loss subplots
plt.show()


## Step 7: Evaluate Model

In [ ]:
# Evaluate on validation set
# Returns the loss and accuracy for the validation data
val_loss, val_acc = model.evaluate(val_generator)

# Print the validation accuracy as a percentage with 2 decimal places
print(f"Validation Accuracy: {val_acc*100:.2f}%")


In [ ]:
# =======================
# Compute Precision, Recall, F1-Score
# =======================

from sklearn.metrics import classification_report
import numpy as np

# Reset the generator so predictions are consistent
val_generator.reset()

# Predict probabilities for all validation images
y_pred_prob = model.predict(val_generator)

# Convert probabilities to class indices
y_pred_classes = np.argmax(y_pred_prob, axis=1)

# True class indices from generator
y_true = val_generator.classes

# Class labels
class_labels = list(val_generator.class_indices.keys())

# Generate classification report
report = classification_report(
    y_true,                # True labels
    y_pred_classes,        # Predicted labels
    target_names=class_labels  # Class names
)

print("=== Validation Classification Report ===")
print(report)


In [ ]:
# Confusion matrix
# Compute the confusion matrix using true labels and predicted class indices
cm = confusion_matrix(y_true, y_pred_classes)

# Create a new figure for the plot with a large size
plt.figure(figsize=(12,10))

# Plot the confusion matrix as a heatmap
# annot=False means numbers are not shown inside each cell
# fmt="d" specifies integer format
# cmap="Blues" sets the color map to shades of blue
sns.heatmap(cm, annot=False, fmt="d", cmap="Reds")

# Set x-axis label
plt.xlabel("Predicted")

# Set y-axis label
plt.ylabel("True")

# Set the title of the plot
plt.title("Validation Confusion Matrix")

# Display the plot
plt.show()


In [ ]:
# Evaluate model on test set
test_loss, test_acc = model.evaluate(test_generator)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc * 100:.2f}%")


In [ ]:
class_names = list(test_generator.class_indices.keys())
print("=== Testing Classification Report ===")
print(classification_report(
    y_true,
    y_pred_classes,
    target_names=class_names
))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(12,10))
sns.heatmap(cm, cmap="Blues", annot=False)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Test Confusion Matrix")
plt.show()


In [ ]:
# Define how many test images to display
num_samples = 12

# Create a figure with specified size for plotting images
plt.figure(figsize=(18,10))

# Loop through the number of samples to display
for i in range(num_samples):

    # Load one batch image and its corresponding label
    img, label = test_generator[i]

    # Get the true class name from the one-hot encoded label
    true_label = class_names[np.argmax(label[0])]

    # Get the predicted class name using model predictions
    pred_label = class_names[y_pred_classes[i]]

    # Check whether the prediction is correct
    correct = (true_label == pred_label)

    # Set title color to green if correct, else red
    title_color = "green" if correct else "red"

    # Define status text based on correctness
    status = "✔ Correct" if correct else "✘ Wrong"

    # Create a subplot for each image
    plt.subplot(3, 4, i + 1)

    # Display the test image
    plt.imshow(img[0])

    # Add title with true label, predicted label, and correctness status
    plt.title(
        f"True: {true_label}\nPred: {pred_label}\n{status}",
        color=title_color,
        fontsize=11
    )

    # Remove axis ticks for better visualization
    plt.axis("off")

# Adjust spacing between subplots
plt.tight_layout()

# Display the plotted images
plt.show()
